Notebook written while developing some evaluation metrics

In [11]:
import numpy as np


true_vec = np.array([4, 2, 3])
estimated_vec = np.array([3, 2, 4])
n_vec = np.array([0, 0, 1])


error = true_vec - estimated_vec

print('Error magnitude', np.linalg.norm(error))

true_vec_los = np.dot(true_vec, n_vec)
estimated_vec_los = np.dot(estimated_vec, n_vec)

los_error = true_vec_los - estimated_vec_los
print('Line-of-sight error magnitude', np.linalg.norm(los_error))

transverse_vec = true_vec - true_vec_los * n_vec
estimated_transverse_vec = estimated_vec - estimated_vec_los * n_vec
transverse_error = transverse_vec - estimated_transverse_vec
print('Transverse error magnitude', np.linalg.norm(transverse_error))

Error magnitude 1.4142135623730951
Line-of-sight error magnitude 1.0
Transverse error magnitude 1.0


In [ ]:
np.dot(true_vec, n_vec)

np.int64(3)

In [ ]:


def error_decomposition(true_vec, estimated_vec, n_vec):

    n_vec = n_vec / np.linalg.norm(n_vec) # ensure n_vec is a unit vector

    error = true_vec - estimated_vec
    error_magnitude = np.linalg.norm(error)

    los_error = np.dot(error, n_vec)
    transverse_error = np.linalg.norm(error - los_error*n_vec)

    rel_error_magnitude = error_magnitude / np.linalg.norm(true_vec)
    rel_los_error = los_error / np.linalg.norm(true_vec)
    rel_transverse_error = transverse_error / np.linalg.norm(true_vec)

    return error_magnitude, rel_error_magnitude, np.abs(los_error), np.abs(rel_los_error), np.abs(transverse_error), np.abs(rel_transverse_error)


In [17]:
import numpy as np

def _check(true_vec, est_vec, n_vec, label):
    true_vec = np.array(true_vec, float)
    est_vec  = np.array(est_vec, float)
    n_vec    = np.array(n_vec, float)
    em, rel_em, los, rel_los, trans, rel_trans = error_decomposition(true_vec, est_vec, n_vec)

    # abs() applied on return, so these are all >= 0
    assert los >= 0 and rel_los >= 0 and trans >= 0 and rel_trans >= 0

    # orthogonal split => Pythagoras, absolute and relative
    assert np.isclose(em**2,     los**2     + trans**2),     f"{label}: abs Pythagoras failed"
    assert np.isclose(rel_em**2, rel_los**2 + rel_trans**2), f"{label}: rel Pythagoras failed"

    print(f"{label:16s} |err|={em:.4f} ({rel_em:.2%}) | "
          f"los={los:.4f} ({rel_los:.2%}) | trans={trans:.4f} ({rel_trans:.2%})")
    return em, rel_em, los, rel_los, trans, rel_trans

# 1) known example -> |err|=1.4142, los=1, trans=1, rel ≈ 26.3% / 18.6% / 18.6%
_check([4, 2, 3], [3, 2, 4], [0, 0, 1], "example")

# 2) n_vec need not be unit-length -> same answer (normalized inside the function)
r1 = error_decomposition(np.array([4.,2,3]), np.array([3.,2,4]), np.array([0.,0,1]))
r2 = error_decomposition(np.array([4.,2,3]), np.array([3.,2,4]), np.array([0.,0,9]))
assert np.allclose(r1, r2), "n_vec normalization not applied"
print("n_vec normalization: OK")

# 3) error fully ALONG the sightline -> transverse == 0
_check([1, 1, 3], [1, 1, 1], [0, 0, 1], "pure LOS")        # error = [0,0,2]

# 4) error fully TRANSVERSE -> los == 0
_check([3, 1, 5], [1, 1, 5], [0, 0, 1], "pure transverse")  # error = [2,0,0]

# 5) fuzz: Pythagoras must hold for arbitrary inputs
rng = np.random.default_rng(0)
for _ in range(2000):
    t, e, n = rng.normal(size=3), rng.normal(size=3), rng.normal(size=3)
    em, rel_em, los, rel_los, trans, rel_trans = error_decomposition(t, e, n)
    assert np.isclose(em**2, los**2 + trans**2)
    assert np.isclose(rel_em**2, rel_los**2 + rel_trans**2)
print("all checks passed ✓")


example          |err|=1.4142 (26.26%) | los=1.0000 (18.57%) | trans=1.0000 (18.57%)
n_vec normalization: OK
pure LOS         |err|=2.0000 (60.30%) | los=2.0000 (60.30%) | trans=0.0000 (0.00%)
pure transverse  |err|=2.0000 (33.81%) | los=0.0000 (0.00%) | trans=2.0000 (33.81%)
all checks passed ✓


Okay nice... now let's think about decomposing in spherical coordinates 

In [ ]:
x = np.array([1, 2, 3])
a = np.array([4, 5, 6])
predicted_a = np.array([4.1, 5.1, 6.1])
R = np.sqrt(x[0]**2 + x[1]**2) # cylindrical coordinates


e_R = np.array([x[0]/R, x[1]/R, 0]) # unit vector in R direction
e_phi = np.array([-x[1]/R, x[0]/R, 0]) # unit vector in phi direction
e_z = np.array([0, 0, 1]) # unit vector in z direction

a_R = np.dot(a, e_R)
a_phi = np.dot(a, e_phi)
a_z = np.dot(a, e_z)

predicted_a_R = np.dot(predicted_a, e_R)
predicted_a_phi = np.dot(predicted_a, e_phi)
predicted_a_z = np.dot(predicted_a, e_z)
